# Profiler output (Chrome / TensorBoard trace)

If you train with profiling enabled:

```bash
python scripts/train.py --output-dir runs/exp1 --profile
```

the run writes under `OUTPUT_DIR/profile/`:

- **TensorBoard trace** fragments (via `tensorboard_trace_handler`) — open with TensorBoard: `tensorboard --logdir OUTPUT_DIR/profile`
- **`profiler_summary.txt`** — text table from `Profiler.key_averages()`

You can also load the trace in Chrome’s **chrome://tracing** (some PyTorch versions export JSON compatible with that workflow; TensorBoard is the most reliable viewer).

Below: list files and show the summary text if present.

In [1]:
from pathlib import Path

OUTPUT_DIR = Path("../runs/exp2")
prof_dir = OUTPUT_DIR / "profile"

if not prof_dir.is_dir():
    print(f"No profiler directory at {prof_dir.resolve()}")
    print("Train once with: python scripts/train.py --output-dir ... --profile")
else:
    files = sorted(prof_dir.rglob("*"))
    for p in files:
        print(p.relative_to(prof_dir))
    summary = prof_dir / "profiler_summary.txt"
    if summary.is_file():
        print("\n--- profiler_summary.txt ---\n")
        print(summary.read_text()[:8000])

No profiler directory at /home/nightbaron/1/work/learning/mnist-former/runs/exp2/profile
Train once with: python scripts/train.py --output-dir ... --profile


## Optional: short profile inside the notebook

Runs a few steps with `torch.profiler` on a single batch (does not replace a full training run).

In [2]:
import tempfile
from pathlib import Path

import torch
import torch.nn as nn

from mnist_former.config import ModelConfig, TrainConfig
from mnist_former.data import get_dataloaders
from mnist_former.model import FashionMNISTViT
from mnist_former.profiling import profile_training_steps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tc = TrainConfig(batch_size=32, num_workers=0)
train_loader, _, _ = get_dataloaders(tc)
xb, yb = next(iter(train_loader))
xb = xb.to(device)
yb = yb.to(device)

model = FashionMNISTViT(ModelConfig()).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
crit = nn.CrossEntropyLoss()


def train_step():
    opt.zero_grad(set_to_none=True)
    loss = crit(model(xb), yb)
    loss.backward()
    opt.step()


with tempfile.TemporaryDirectory() as td:
    out = Path(td)
    profile_training_steps(device, train_step, out, warmup=1, active=4)
    print("Profiler summary (notebook temp dir):")
    print((out / "profiler_summary.txt").read_text()[:4000])

/home/nightbaron/miniconda3/envs/mnist-former/lib/python3.11/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


Profiler summary (notebook temp dir):
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.00%       0.000us         0.00%       0.000us       0.000us      10.207ms       520.88%      10.207ms       2.552ms           0 B   